# Day 052 Solution — AI API with FastAPI

Section 4, Day 2. Builds the full AI API with `build_api`, exercises every route with `TestClient` (in-process — no server), then generates a real, runnable `main.py`. This notebook never starts uvicorn; it verifies the app and the generated file. Launch the service with `uvicorn main:app`.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from starlette.testclient import TestClient
import ollama


class ChatRequest(BaseModel):
    """Request body for the chat endpoints."""
    message: str = Field(min_length=1, description='User message for the model')
    temperature: float = Field(default=0.7, ge=0.0, le=1.0)


class ChatResponse(BaseModel):
    """Response body returned by the chat endpoints."""
    reply: str
    model: str


class HealthResponse(BaseModel):
    """Response body for the health check."""
    status: str
    model: str


PROMPT_TEMPLATES = {
    'summary':  'Summarize the following topic in two sentences: {topic}',
    'explain':  'Explain {topic} to a complete beginner.',
    'critique': 'List three criticisms of {topic}.',
}


def run_model(model: str, prompt: str, temperature: float = 0.7) -> str:
    """Call Ollama once and return the reply text. Raises on model error."""
    resp = ollama.chat(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        options={'temperature': temperature},
    )
    return resp['message']['content'].strip()


def build_api(model: str = 'llama3.2') -> FastAPI:
    """Assemble the complete AI API: health, templates, chat, and templated chat."""
    app = FastAPI(title='AI API', version='1.0.0')

    @app.get('/health', response_model=HealthResponse)
    def health():
        return HealthResponse(status='ok', model=model)

    @app.get('/templates')
    def list_templates():
        return {'templates': list(PROMPT_TEMPLATES.keys())}

    @app.post('/chat', response_model=ChatResponse)
    def chat(req: ChatRequest):
        try:
            return ChatResponse(reply=run_model(model, req.message, req.temperature),
                                model=model)
        except Exception as e:
            raise HTTPException(status_code=503, detail=f'Model unavailable: {e}')

    @app.post('/render/{name}', response_model=ChatResponse)
    def render_chat(name: str, req: ChatRequest):
        if name not in PROMPT_TEMPLATES:
            raise HTTPException(status_code=404, detail=f'template {name!r} not found')
        prompt = PROMPT_TEMPLATES[name].format(topic=req.message)
        try:
            return ChatResponse(reply=run_model(model, prompt, req.temperature),
                                model=model)
        except Exception as e:
            raise HTTPException(status_code=503, detail=f'Model unavailable: {e}')

    return app

## Step 1 — Build the App and Test It In-Process

In [ ]:
app = build_api()
client = TestClient(app)

print('GET /health   ->', client.get('/health').json())
print('GET /templates->', client.get('/templates').json())
assert client.get('/health').status_code == 200

## Step 2 — Call the AI Endpoints

In [ ]:
r = client.post('/chat', json={'message': 'Say hello in exactly three words.'})
print('POST /chat  ->', r.status_code, r.json())
assert r.status_code == 200 and len(r.json()['reply']) > 0

r = client.post('/render/summary', json={'message': 'FastAPI'})
print('POST /render/summary ->', r.status_code)
print('  reply:', r.json()['reply'][:200])
assert r.status_code == 200 and len(r.json()['reply']) > 0

## Step 3 — Error Paths: 422 and 404

In [ ]:
bad = client.post('/chat', json={'temperature': 0.5})  # missing message
print('missing message ->', bad.status_code)
assert bad.status_code == 422

missing = client.post('/render/does-not-exist', json={'message': 'x'})
print('unknown template ->', missing.status_code)
assert missing.status_code == 404
print('Validation (422) and not-found (404) behave correctly.')

## Step 4 — Generate the Runnable main.py

In [ ]:
from pathlib import Path

# The full FastAPI app source (models + templates + run_model + build_api +
# a uvicorn entry point). Embedded as a string so we can write it to a real
# file — the runnable deliverable you launch with `uvicorn main:app`.
_MAIN_SRC = 'import warnings\nwarnings.filterwarnings(\'ignore\')\nfrom fastapi import FastAPI, HTTPException\nfrom pydantic import BaseModel, Field\nimport ollama\n\n\nclass ChatRequest(BaseModel):\n    """Request body for the chat endpoints."""\n    message: str = Field(min_length=1, description=\'User message for the model\')\n    temperature: float = Field(default=0.7, ge=0.0, le=1.0)\n\n\nclass ChatResponse(BaseModel):\n    """Response body returned by the chat endpoints."""\n    reply: str\n    model: str\n\n\nclass HealthResponse(BaseModel):\n    """Response body for the health check."""\n    status: str\n    model: str\n\n\nPROMPT_TEMPLATES = {\n    \'summary\':  \'Summarize the following topic in two sentences: {topic}\',\n    \'explain\':  \'Explain {topic} to a complete beginner.\',\n    \'critique\': \'List three criticisms of {topic}.\',\n}\n\n\ndef run_model(model: str, prompt: str, temperature: float = 0.7) -> str:\n    """Call Ollama once and return the reply text. Raises on model error."""\n    resp = ollama.chat(\n        model=model,\n        messages=[{\'role\': \'user\', \'content\': prompt}],\n        options={\'temperature\': temperature},\n    )\n    return resp[\'message\'][\'content\'].strip()\n\n\ndef build_api(model: str = \'llama3.2\') -> FastAPI:\n    """Assemble the complete AI API: health, templates, chat, and templated chat."""\n    app = FastAPI(title=\'AI API\', version=\'1.0.0\')\n\n    @app.get(\'/health\', response_model=HealthResponse)\n    def health():\n        return HealthResponse(status=\'ok\', model=model)\n\n    @app.get(\'/templates\')\n    def list_templates():\n        return {\'templates\': list(PROMPT_TEMPLATES.keys())}\n\n    @app.post(\'/chat\', response_model=ChatResponse)\n    def chat(req: ChatRequest):\n        try:\n            return ChatResponse(reply=run_model(model, req.message, req.temperature),\n                                model=model)\n        except Exception as e:\n            raise HTTPException(status_code=503, detail=f\'Model unavailable: {e}\')\n\n    @app.post(\'/render/{name}\', response_model=ChatResponse)\n    def render_chat(name: str, req: ChatRequest):\n        if name not in PROMPT_TEMPLATES:\n            raise HTTPException(status_code=404, detail=f\'template {name!r} not found\')\n        prompt = PROMPT_TEMPLATES[name].format(topic=req.message)\n        try:\n            return ChatResponse(reply=run_model(model, prompt, req.temperature),\n                                model=model)\n        except Exception as e:\n            raise HTTPException(status_code=503, detail=f\'Model unavailable: {e}\')\n\n    return app\n\n\napp = build_api()\n\n\nif __name__ == \'__main__\':\n    import uvicorn\n    uvicorn.run(app, host=\'0.0.0.0\', port=8000)\n'


def write_api_app(path: str = 'main.py') -> str:
    """Write the self-contained FastAPI app to `path` and return the path."""
    Path(path).write_text(_MAIN_SRC, encoding='utf-8')
    return path

In [ ]:
path = write_api_app('main.py')
src = open(path, encoding='utf-8').read()
print(f'Wrote {path} ({len(src)} chars)')

assert 'from fastapi import FastAPI' in src
assert 'def build_api' in src
assert 'app = build_api()' in src
assert 'uvicorn' in src
compile(src, 'main.py', 'exec')  # must be valid Python
print('main.py verified: imports FastAPI, defines build_api, exposes app, compiles.')

## Step 5 — Preview the Entry Point

In [ ]:
tail = src[src.index('app = build_api()'):]
print(tail)

print('\nTo launch the API, run in a terminal:')
print('    uvicorn main:app --reload')
print('Then open http://localhost:8000/docs for interactive docs.')
print('\nDay 52 — AI API complete! 🎉')